# putEMG — Efficient Deep Learning Model (8-Channel LOSO)

Retrains the same LOSO cross-subject evaluation as `experiments/cross_subject/deep_learning/`
but using only the **8 representative channels** selected by feature-based agglomerative clustering
(see `experiments/clustering/clustering_features.ipynb`).

**Goal:** match or closely approach the 24-channel baseline accuracy with one-third the
electrode count — demonstrating practical deployability in a wearable device.

| Input | 24-channel baseline | This notebook |
|-------|--------------------|--------------|
| Shape | `(N, 1, 24, 1500)` | `(N, 1, 8, 1500)` |
| Source | `load_all_subjects` | same + channel slice |

**Prerequisites:**
1. `data_preprocessing/driver.ipynb` — per-subject `.mat` files
2. `experiments/clustering/clustering_features.ipynb` — `feat_representative_channels.npy`

Set `MODEL_TYPE` in the config cell to switch between `EMG_TCN`, `EEGNet`, `ShallowConvNet`.
Completed folds are skipped automatically — safe to stop and resume.

In [1]:
print(5)

5


In [2]:
import os
import sys
import numpy as np
import datetime
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau

In [3]:
# ── Shared utilities (src/) ────────────────────────────────────────────────────
sys.path.append(os.path.abspath('../../../'))

import src.deep_learning_models as dlm
from src.emg_loader import load_all_subjects, make_loso_train_val_test

In [4]:
# ── Load channel selection from clustering notebook ────────────────────────────
CLUSTERING_DIR = os.path.abspath('../../clustering')
channels_path  = os.path.join(CLUSTERING_DIR, 'feat_representative_channels.npy')

if not os.path.exists(channels_path):
    raise FileNotFoundError(
        f'Channel selection file not found: {channels_path}\n'
        'Run experiments/clustering/clustering_features.ipynb first.'
    )

CHANNELS = np.load(channels_path)   # (8,) — 0-indexed channel indices
N_CH     = len(CHANNELS)
print(f'Using {N_CH} channels (0-indexed): {list(CHANNELS)}')
print(f'(1-indexed): {[c + 1 for c in CHANNELS]}')

Using 8 channels (0-indexed): [0, 3, 6, 14, 16, 18, 20, 22]
(1-indexed): [1, 4, 7, 15, 17, 19, 21, 23]


In [5]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

def train(model, train_loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    for X, y in train_loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(X), y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(train_loader)


def evaluate(model, loader, device):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            correct += (model(X).argmax(dim=1) == y).sum().item()
            total   += y.size(0)
    return correct / total

In [6]:
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: mps


In [7]:
_notebook_dir = os.path.abspath(os.getcwd())

# ── Config ────────────────────────────────────────────────────────────────────
DATA_DIR    = '/Volumes/KRIS/data/UG_per_subject'
MODEL_TYPE  = 'EMG_TCN'   # 'EEGNet' | 'ShallowConvNet'
WEIGHTS_DIR = os.path.join(_notebook_dir, 'weights', MODEL_TYPE)
RESULTS_DIR = os.path.join(_notebook_dir, 'results')
LOG_PATH    = os.path.join(RESULTS_DIR, f'results_log_{MODEL_TYPE}.txt')

BATCH_SIZE = 16
VAL_FRAC   = 0.10
MAX_EPOCHS = 20
PATIENCE   = 5
MIN_DELTA  = 0.002
LR         = 1e-3
DROPOUT    = 0.1

os.makedirs(WEIGHTS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# ── Load and slice to 8 channels ──────────────────────────────────────────────
# load_all_subjects → list of (name, X, y)  where X: (N_reps, 1, 24, 1500)
# Slice axis 2 to keep only the 8 representative channels → (N_reps, 1, 8, 1500)
_subjects_full = load_all_subjects(DATA_DIR)
subjects = [(name, X[:, :, CHANNELS, :], y) for name, X, y in _subjects_full]

print(f'\nInput shape after channel selection: {subjects[0][1].shape}')
print(f'Weights → {WEIGHTS_DIR}')
print(f'Log     → {LOG_PATH}')

Loading 44 subject file(s) from: /Volumes/KRIS/data/UG_per_subject

  → emg_gestures_03_U.mat  (280 samples)
  → emg_gestures_04_U.mat  (280 samples)
  → emg_gestures_05_U.mat  (280 samples)
  → emg_gestures_06_U.mat  (280 samples)
  → emg_gestures_07_U.mat  (280 samples)
  → emg_gestures_08_U.mat  (280 samples)
  → emg_gestures_09_U.mat  (280 samples)
  → emg_gestures_10_U.mat  (280 samples)
  → emg_gestures_11_U.mat  (280 samples)
  → emg_gestures_12_U.mat  (280 samples)
  → emg_gestures_13_U.mat  (280 samples)
  → emg_gestures_14_U.mat  (280 samples)
  → emg_gestures_15_U.mat  (280 samples)
  → emg_gestures_16_U.mat  (280 samples)
  → emg_gestures_17_U.mat  (280 samples)
  → emg_gestures_18_U.mat  (280 samples)
  → emg_gestures_19_U.mat  (280 samples)
  → emg_gestures_20_U.mat  (280 samples)
  → emg_gestures_22_U.mat  (280 samples)
  → emg_gestures_23_U.mat  (280 samples)
  → emg_gestures_24_U.mat  (280 samples)
  → emg_gestures_25_U.mat  (280 samples)
  → emg_gestures_26_U.mat  (28

---
## LOSO Training Loop

Identical protocol to `experiments/cross_subject/deep_learning/` — only the input
shape changes from `(N, 1, 24, 1500)` to `(N, 1, 8, 1500)`.

- **Test** — 1 held-out subject, never seen during training
- **Train / Val** — all other 43 subjects, split 90/10 (stratified by class, seeded)
- Checkpointing: fold skipped if `weights/<MODEL_TYPE>/<MODEL_TYPE>_<id>.pt` already exists

In [8]:
MODEL_MAP = {
    'EMG_TCN':        dlm.EMG_TCN,
    'EEGNet':         dlm.EEGNet,
    'ShallowConvNet': dlm.ShallowConvNet,
}

def subject_id(filename):
    return filename.replace('emg_gestures_', '').replace('_U.mat', '')


def weight_path(subject_name):
    return os.path.join(WEIGHTS_DIR, f'{MODEL_TYPE}_{subject_id(subject_name)}.pt')


def update_log():
    checkpoints = []
    for fname in sorted(os.listdir(WEIGHTS_DIR)):
        if not fname.endswith('.pt'):
            continue
        ckpt = torch.load(os.path.join(WEIGHTS_DIR, fname), map_location='cpu', weights_only=False)
        if 'subject' in ckpt:
            checkpoints.append(ckpt)

    if not checkpoints:
        return

    accs   = [c['test_acc'] * 100 for c in checkpoints]
    n_done = len(checkpoints)

    lines = [
        f'putEMG — {MODEL_TYPE} LOSO Results  (8-channel efficient model)',
        '=' * 68,
        f"{'Subject':<28} {'Test Acc':>9}  {'Val Acc':>9}  {'Epoch':>6}  {'Date'}",
        '-' * 68,
    ]
    for c in checkpoints:
        lines.append(
            f"{c['subject']:<28} {c['test_acc']*100:>8.2f}%  "
            f"{c['val_acc']*100:>8.2f}%  {c['best_epoch']:>6}  {c['date']}"
        )
    lines += [
        '=' * 68,
        f"Mean: {np.mean(accs):.2f}%  ±  {np.std(accs):.2f}%  "
        f"({n_done} / {len(subjects)} folds complete)",
    ]

    with open(LOG_PATH, 'w') as f:
        f.write('\n'.join(lines) + '\n')

    print(f'Log updated → {LOG_PATH}  ({n_done}/{len(subjects)} folds)')

In [9]:
for test_idx, (test_name, _, _) in enumerate(subjects):
    wpath = weight_path(test_name)

    if os.path.exists(wpath):
        print(f'[SKIP] {test_name}  — checkpoint found')
        continue

    print(f"\n{'='*60}")
    print(f'  Fold {test_idx+1}/{len(subjects)}  —  test: {test_name}')
    print(f"{'='*60}")

    train_loader, val_loader, test_loader = make_loso_train_val_test(
        subjects, test_idx, val_frac=VAL_FRAC, batch_size=BATCH_SIZE
    )

    # num_channels=N_CH tells each model to expect 8 input channels instead of 24
    model     = MODEL_MAP[MODEL_TYPE](num_channels=N_CH, dropout_rate=DROPOUT).to(device)
    optimizer = optim.Adam(model.parameters(), lr=LR)
    scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5, min_lr=1e-6)
    criterion = nn.CrossEntropyLoss()

    best_val_acc = float('-inf')
    best_state   = None
    best_epoch   = 0
    bad_epochs   = 0

    for epoch in range(MAX_EPOCHS):
        tr_loss = train(model, train_loader, criterion, optimizer, device)
        val_acc = evaluate(model, val_loader, device)
        curr_lr = optimizer.param_groups[0]['lr']

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state   = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            best_epoch   = epoch + 1

        scheduler.step(val_acc)
        bad_epochs = 0 if val_acc >= (best_val_acc - MIN_DELTA) else bad_epochs + 1

        print(f'  Epoch {epoch+1:3d}: loss={tr_loss:.4f}  val={val_acc*100:.2f}%  '
              f'best={best_val_acc*100:.2f}%  lr={curr_lr:.2e}')

        if bad_epochs >= PATIENCE:
            print(f'  Early stopping at epoch {epoch+1}.')
            break

    model.load_state_dict(best_state)
    test_acc = evaluate(model, test_loader, device)
    print(f'\n  Test accuracy: {test_acc*100:.2f}%')

    torch.save({
        'subject':    test_name,
        'test_acc':   test_acc,
        'val_acc':    best_val_acc,
        'best_epoch': best_epoch,
        'channels':   list(CHANNELS),
        'n_channels': N_CH,
        'dropout':    DROPOUT,
        'state_dict': best_state,
        'date':       datetime.date.today().isoformat(),
    }, wpath)

    update_log()

print(f"\n{'='*60}")
print('  All folds complete.')
print(f"{'='*60}")
update_log()


  Fold 1/44  —  test: emg_gestures_03_U.mat
  Test  : emg_gestures_03_U.mat  (280 reps)
  Train : 10836 reps  |  Val: 1204 reps  (43 subjects, 90/10 split)
  Epoch   1: loss=1.0120  val=68.60%  best=68.60%  lr=1.00e-03
  Epoch   2: loss=0.7681  val=75.33%  best=75.33%  lr=1.00e-03
  Epoch   3: loss=0.6781  val=77.08%  best=77.08%  lr=1.00e-03
  Epoch   4: loss=0.6027  val=78.74%  best=78.74%  lr=1.00e-03
  Epoch   5: loss=0.5745  val=80.90%  best=80.90%  lr=1.00e-03
  Epoch   6: loss=0.5140  val=80.40%  best=80.90%  lr=1.00e-03
  Epoch   7: loss=0.5013  val=81.15%  best=81.15%  lr=1.00e-03
  Epoch   8: loss=0.4717  val=80.48%  best=81.15%  lr=1.00e-03
  Epoch   9: loss=0.4469  val=84.22%  best=84.22%  lr=1.00e-03
  Epoch  10: loss=0.4303  val=84.88%  best=84.88%  lr=1.00e-03
  Epoch  11: loss=0.4125  val=88.04%  best=88.04%  lr=1.00e-03
  Epoch  12: loss=0.3952  val=83.22%  best=88.04%  lr=1.00e-03
  Epoch  13: loss=0.3776  val=87.62%  best=88.04%  lr=1.00e-03
  Epoch  14: loss=0.3525

KeyboardInterrupt: 

---
## Results

In [ ]:
if os.path.exists(LOG_PATH):
    with open(LOG_PATH) as f:
        print(f.read())
else:
    print('No results yet — run the training loop first.')

In [ ]:
# ── Per-subject bar chart ──────────────────────────────────────────────────────
checkpoints = []
for fname in sorted(os.listdir(WEIGHTS_DIR)):
    if not fname.endswith('.pt'):
        continue
    ckpt = torch.load(os.path.join(WEIGHTS_DIR, fname), map_location='cpu', weights_only=False)
    if 'subject' in ckpt:
        checkpoints.append(ckpt)

if checkpoints:
    sids = [subject_id(c['subject']) for c in checkpoints]
    accs = [c['test_acc'] * 100 for c in checkpoints]
    mean = np.mean(accs)

    fig, ax = plt.subplots(figsize=(max(10, len(sids) * 0.45), 4))
    ax.bar(sids, accs, color='steelblue')
    ax.axhline(mean, color='tomato', linestyle='--', linewidth=1.5, label=f'Mean {mean:.1f}%')
    ax.set_ylim(0, 105)
    ax.set_xlabel('Test Subject')
    ax.set_ylabel('Test Accuracy (%)')
    ax.set_title(f'{MODEL_TYPE} LOSO — 8-Channel Efficient Model  ({len(checkpoints)}/{len(subjects)} folds)')
    ax.legend()
    plt.xticks(rotation=45, ha='right', fontsize=8)
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Compare 8-channel vs 24-channel baseline ───────────────────────────────────
# Loads the corresponding log from cross_subject/deep_learning/ for side-by-side comparison
baseline_log = os.path.abspath(
    f'../../cross_subject/deep_learning/results/results_log_{MODEL_TYPE}.txt'
)

if os.path.exists(baseline_log) and checkpoints:
    # Load 24-channel baseline checkpoints
    baseline_weights = os.path.abspath(
        f'../../cross_subject/deep_learning/weights/{MODEL_TYPE}'
    )
    baseline_ckpts = []
    if os.path.isdir(baseline_weights):
        for fname in sorted(os.listdir(baseline_weights)):
            if not fname.endswith('.pt'):
                continue
            ckpt = torch.load(os.path.join(baseline_weights, fname),
                              map_location='cpu', weights_only=False)
            if 'subject' in ckpt:
                baseline_ckpts.append(ckpt)

    if baseline_ckpts:
        # Align by subject name for a fair per-subject comparison
        eff_map  = {subject_id(c['subject']): c['test_acc'] * 100 for c in checkpoints}
        base_map = {subject_id(c['subject']): c['test_acc'] * 100 for c in baseline_ckpts}
        common   = sorted(set(eff_map) & set(base_map))

        eff_accs  = [eff_map[s]  for s in common]
        base_accs = [base_map[s] for s in common]
        drops     = [b - e for b, e in zip(base_accs, eff_accs)]

        print(f'\n{MODEL_TYPE} — 24-channel vs 8-channel comparison  ({len(common)} subjects)')
        print(f'  24-channel mean : {np.mean(base_accs):.2f}%')
        print(f'   8-channel mean : {np.mean(eff_accs):.2f}%')
        print(f'  Accuracy drop   : {np.mean(drops):+.2f} pp  (mean per subject)')

        x = np.arange(len(common))
        w = 0.38
        fig, ax = plt.subplots(figsize=(max(12, len(common) * 0.5), 4))
        ax.bar(x - w/2, base_accs, w, label='24-channel baseline', color='steelblue')
        ax.bar(x + w/2, eff_accs,  w, label='8-channel efficient',  color='darkorange')
        ax.set_xticks(x)
        ax.set_xticklabels(common, rotation=45, ha='right', fontsize=8)
        ax.set_ylim(0, 105)
        ax.set_xlabel('Test Subject')
        ax.set_ylabel('Test Accuracy (%)')
        ax.set_title(f'{MODEL_TYPE}: 24-channel vs 8-channel LOSO')
        ax.legend()
        plt.tight_layout()
        plt.show()
else:
    print('Run training first, or ensure cross_subject/deep_learning results exist for comparison.')